# Same Anwser Anomaly

In [1]:
import pandas as pd
import pyodbc  
import matplotlib.pyplot as plt
import numpy as np

## Import Data from csv

In [4]:
# paq_question = pd.read_csv("../../../decoded_data/PAQ/FactQuestionPAQ.csv")
paq_question = pd.read_csv('../csv/preprocessed_data.csv')

## Data Inspection

In [5]:
paq_question.head()

,TestKey,QuestionKey,LeftStatement,RightStatement,AnswerVal,CandidateKey,Year,Month,Day
0,166959,1,0,18,1,38114181,2019,1,2
1,166959,2,42,60,5,38114181,2019,1,2
2,166959,3,48,0,5,38114181,2019,1,2
3,166959,4,54,30,5,38114181,2019,1,2
4,166959,5,30,42,5,38114181,2019,1,2


In [6]:
paq_question.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502560 entries, 0 to 502559
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   TestKey         502560 non-null  int64
 1   QuestionKey     502560 non-null  int64
 2   LeftStatement   502560 non-null  int64
 3   RightStatement  502560 non-null  int64
 4   AnswerVal       502560 non-null  int64
 5   CandidateKey    502560 non-null  int64
 6   Year            502560 non-null  int64
 7   Month           502560 non-null  int64
 8   Day             502560 non-null  int64
dtypes: int64(9)
memory usage: 34.5 MB


## Data Preparation

The only data cleaning that needs to happen here is getting rid of the coloms we won't need in this anomaly detection. There seem to be no empty fields or suspicious values to worry about. 

In [7]:
df = paq_question[["QuestionKey", "AnswerVal", "TestKey"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502560 entries, 0 to 502559
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   QuestionKey  502560 non-null  int64
 1   AnswerVal    502560 non-null  int64
 2   TestKey      502560 non-null  int64
dtypes: int64(3)
memory usage: 11.5 MB


# Same Anwsers on Test Anomaly

In [8]:
# Groepeer de data per CandidateId, en behoud ook InstanceId
all_test_answers = df.groupby(['TestKey']).agg({
    'AnswerVal': list
}).reset_index()

# Voeg alle antwoorden samen in een enkele array
all_test_answers['AllAnswers'] = all_test_answers.apply(
    lambda row: np.array(row['AnswerVal']),
    axis=1
)

## Detection function

In [9]:
def detect_same_answers(TestKey, print_result=False):
    answers = all_test_answers[all_test_answers['TestKey'] == TestKey].reset_index().AllAnswers[0]

    ans_count = {
        0:0, 
        1:0, 
        2:0,
        3:0,
        4:0,
        5:0
        }
    total_count = 0

    for i in answers: 
        ans_count[i] += 1
        total_count += 1

    max_count = max(ans_count.values())
    most_common_ans = max(ans_count, key=ans_count.get)
    perc = (max_count/total_count) * 100
    if print_result:
        print(f"Candidate {TestKey} answered {most_common_ans} on {perc}% of the questions.")
    return perc


In [10]:
detect_same_answers(166962, print_result=True)

Candidate 166962 answered 5 on 68.88888888888889% of the questions.


68.88888888888889

In [11]:
detect_same_answers(166963, print_result=True)

Candidate 166963 answered 4 on 32.22222222222222% of the questions.


32.22222222222222

In [12]:
detect_same_answers(166964, print_result=True)

Candidate 166964 answered 4 on 36.666666666666664% of the questions.


36.666666666666664

## Exporting Data with Anomaly Check

In [13]:
all_test_answers['SameAnswerPercentage'] = all_test_answers['TestKey'].apply(detect_same_answers)
all_test_answers

,TestKey,AnswerVal,AllAnswers,SameAnswerPercentage
0,166959,"[1, 5, 5, 5, 5, 5, 5, 3, 5, 1, 4, 5, 1, 5, 1, ...","[1, 5, 5, 5, 5, 5, 5, 3, 5, 1, 4, 5, 1, 5, 1, ...",44.444444
1,166960,"[2, 2, 1, 4, 1, 4, 4, 5, 4, 4, 2, 1, 4, 2, 1, ...","[2, 2, 1, 4, 1, 4, 4, 5, 4, 4, 2, 1, 4, 2, 1, ...",35.555556
2,166961,"[3, 4, 3, 3, 3, 4, 4, 3, 4, 1, 3, 3, 3, 3, 2, ...","[3, 4, 3, 3, 3, 4, 4, 3, 4, 1, 3, 3, 3, 3, 2, ...",51.111111
3,166962,"[2, 3, 4, 5, 1, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, ...","[2, 3, 4, 5, 1, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5, ...",68.888889
4,166963,"[3, 1, 2, 3, 2, 5, 1, 4, 5, 5, 5, 2, 1, 1, 1, ...","[3, 1, 2, 3, 2, 5, 1, 4, 5, 5, 5, 2, 1, 1, 1, ...",32.222222
...,...,...,...,...
5581,172540,"[2, 3, 3, 4, 2, 4, 4, 3, 3, 4, 2, 3, 2, 3, 2, ...","[2, 3, 3, 4, 2, 4, 4, 3, 3, 4, 2, 3, 2, 3, 2, ...",45.555556
5582,172541,"[2, 2, 4, 4, 2, 4, 4, 2, 2, 5, 5, 3, 1, 3, 1, ...","[2, 2, 4, 4, 2, 4, 4, 2, 2, 5, 5, 3, 1, 3, 1, ...",24.444444
5583,172542,"[1, 4, 5, 3, 2, 4, 4, 4, 3, 3, 2, 3, 1, 3, 2, ...","[1, 4, 5, 3, 2, 4, 4, 4, 3, 3, 2, 3, 1, 3, 2, ...",40.000000
5584,172543,"[3, 3, 5, 4, 2, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, ...","[3, 3, 5, 4, 2, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, ...",61.111111


In [14]:
all_test_answers.drop(columns=["AllAnswers", "AnswerVal"], inplace=True)
all_test_answers

,TestKey,SameAnswerPercentage
0,166959,44.444444
1,166960,35.555556
2,166961,51.111111
3,166962,68.888889
4,166963,32.222222
...,...,...
5581,172540,45.555556
5582,172541,24.444444
5583,172542,40.000000
5584,172543,61.111111


In [15]:
all_test_answers['is_anomaly'] = all_test_answers.SameAnswerPercentage >= 50
all_test_answers.is_anomaly = all_test_answers.is_anomaly.astype(int)
all_test_answers.head()

,TestKey,SameAnswerPercentage,is_anomaly
0,166959,44.444444,0
1,166960,35.555556,0
2,166961,51.111111,1
3,166962,68.888889,1
4,166963,32.222222,0


In [16]:
all_test_answers.to_csv("../csv/same_answers_test_checked.csv")